# Generative Model Architectures

**Module:** 17 — Image Generation

GANs, autoencoders/VAEs, diffusion, and transformer-based image generators — trade-offs that still shape today's stacks.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain GAN, VAE/AE, diffusion, and transformer image generators
- Compare training objectives and failure modes
- Reason about why latent diffusion + transformers dominate
- Pick an architecture family for a constrained brief


## Architecture family map

```mermaid
flowchart TB
  GAN[GANs] --> DM[Diffusion]
  AE[Autoencoders / VAEs] --> DM
  DM --> TR[Transformers / DiT / AR]
```

| Family | Core idea | Strength | Weakness |
|--------|-----------|----------|----------|
| **GAN** | Generator vs discriminator | Fast, sharp | Mode collapse, unstable |
| **AE / VAE** | Encode→latent→decode | Compression, editing | Blurry if naïve |
| **Diffusion** | Iterative denoise | Quality + control | Many steps → latency |
| **Transformer** | Attention over patches/tokens | Scale, multimodal | Compute hungry |


## GANs

### Definition
A **GAN** trains generator G to fool discriminator D that tells real vs fake.

### Why it matters
Pioneered photoreal synthesis; still in upscalers and fast generators.

### How it works
Minimax: G minimizes D's accuracy on fakes; D maximizes discrimination.

### Intuition
Counterfeiter vs detective — both improve until fakes are hard to spot.

### Pitfalls
- Mode collapse
- Loss oscillations
- Hard precise conditioning without cGAN/StyleGAN care

### When to use
Low-latency generation and mature domain-specific recipes.


In [ ]:
# Demo 1: educational GAN losses
import math

def bce_with_logits(logit: float, target: float) -> float:
    return max(logit, 0) - logit * target + math.log1p(math.exp(-abs(logit)))

def gan_step(d_real: float, d_fake: float) -> dict:
    loss_d = bce_with_logits(d_real, 1.0) + bce_with_logits(d_fake, 0.0)
    loss_g = bce_with_logits(d_fake, 1.0)
    return {"loss_d": round(loss_d, 4), "loss_g": round(loss_g, 4)}

print("healthy-ish", gan_step(2.0, -1.5))
print("generator winning", gan_step(0.1, 2.0))
print("discriminator winning", gan_step(3.0, -3.0))


### Nested GAN topics

| Topic | One-liner |
|-------|-----------|
| DCGAN | Conv G/D template |
| cGAN | Condition on labels/text |
| StyleGAN | Style latents + progressive detail |
| Pix2Pix / CycleGAN | Image-to-image translation |


## Autoencoders and VAEs

### Definition
An **autoencoder** compresses x→z→x̂. A **VAE** makes z probabilistic so you can sample/interpolate.

### Why it matters
VAEs underpin latent diffusion (Stable Diffusion's VAE).

### How it works
Train reconstruction (+ KL for VAEs). Decode z or run another generative model in z-space.

### Intuition
A smooth ZIP file for images — nearby codes mean similar pictures.

### Pitfalls
- Pixel MSE → blur
- Ignoring VAE latent scaling
- Editing entangled z blindly

### When to use
Compression, latent diffusion backbones, representation learning.


In [ ]:
# Demo 2: VAE reparameterization + KL
import random, math

def reparam(mu: float, logvar: float) -> float:
    return mu + math.exp(0.5 * logvar) * random.gauss(0, 1)

def kl_standard_normal(mu: float, logvar: float) -> float:
    return -0.5 * (1 + logvar - mu * mu - math.exp(logvar))

random.seed(0)
print([round(reparam(0.5, -1.0), 3) for _ in range(5)])
print("kl", round(kl_standard_normal(0.5, -1.0), 4))


## Diffusion Models

### Definition
Diffusion learns to reverse gradual noising so pure noise can be iteratively denoised into an image.

### Why it matters
Dominates open and commercial image systems for quality, diversity, and controllability.

### How it works
Forward add noise over T steps; reverse network predicts noise/velocity; sampler iterates to t=0.

### Intuition
A stained-glass window cleaned one wipe at a time.

### Pitfalls
- Too few steps without distillation
- CFG too high → brittle
- Wrong VAE scaling

### When to use
General-purpose txt2img/img2img; base for ControlNet/LoRA ecosystems.


In [ ]:
# Demo 3: 1D forward diffusion intuition
import random

def forward_noisy(x0: float, t: int, beta: float = 0.02) -> float:
    alpha_bar = (1 - beta) ** t
    return (alpha_bar ** 0.5) * x0 + ((1 - alpha_bar) ** 0.5) * random.gauss(0, 1)

random.seed(1)
for t in [0, 10, 50, 100]:
    print(f"t={t:3d}", [round(forward_noisy(2.0, t), 2) for _ in range(3)])


## Transformer-Based Image Generators

### Definition
Transformers generate/denoise images as patch or latent token grids (AR, masked, or DiT).

### Why it matters
Scaling laws and multimodal unification push industry toward transformer backbones.

### How it works
Patchify latents → attention/MLP conditioned on timestep+text → decode.

### Intuition
The same attention engine as LLMs, painting with patches.

### Pitfalls
- Quadratic attention cost
- Naive AR slow at high-res
- 'Transformer' alone ≠ quality

### When to use
Frontier models and multimodal systems with transformer infra.


In [ ]:
# Demo 4: patch token / attention cost sizing
def patch_tokens(height: int, width: int, patch: int = 16, latent_scale: int = 8) -> dict:
    lh, lw = height // latent_scale, width // latent_scale
    ph, pw = lh // patch, lw // patch
    return {"latent_hw": (lh, lw), "tokens": ph * pw, "attn_units": (ph * pw) ** 2}

for size in [(256, 256), (512, 512), (1024, 1024)]:
    print(size, patch_tokens(*size))


## Comparison snapshot

| Need | Prefer |
|------|--------|
| Fast few-step mobile | Distilled diffusion / GAN / small AR |
| Open ControlNet/LoRA ecosystem | Latent diffusion (SD-family) |
| Unified multimodal scale | Transformer / DiT |
| Latent editing | VAE + generative prior on z |


### Try it yourself — Architectures

1. Simulate mode-collapse: generator emits 2 of 10 modes.
2. Compute tokens for 768×768 with patch=2 in latent space (scale=8).
3. Recommend a stack for indie game concept art on one 4090.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `mode collapse` | Generator covers few data modes |
| `DiT` | Diffusion Transformer backbone |
| `reparameterization` | z=μ+σ⊙ε for VAE backprop |
| `score` | Gradient of log-density / denoising target |


### Workshop — Parameter journal — Gen Architectures

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Gen Architectures
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Gen Architectures

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Gen Architectures
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Gen Architectures

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Gen Architectures
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Gen Architectures

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Gen Architectures
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Gen Architectures

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Gen Architectures
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Gen Architectures

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Gen Architectures
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Gen Architectures

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Gen Architectures
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


## Key Takeaways

- Four families still matter: GAN, AE/VAE, diffusion, transformers
- Latent diffusion won open ecosystems; transformers win scaling narratives
- Choose by latency, controllability, and tooling
